# Chapter 6, Exercise 5: A per-dialect error audit of Whisper with Label Studio export

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 6, Exercise 5.** Run a hands-on per-dialect error audit. Take a short (about five-minute) sample of spontaneous Saudi or Gulf speech from SADA, or Egyptian speech from a suitable corpus such as MASC, produce a baseline Whisper transcription, and load the output beside the reference transcript in a text annotation tool such as Label Studio. Manually tag each error, strictly separating genuine acoustic misrecognitions from MSA translation-style shifts where a dialectal word is rendered as its formal equivalent. Report the per-dialect WER and discuss how that single number misrepresents the model's behavior on dialect.

**Note on data.** The official SADA release is distributed through Kaggle (login and licence acceptance required, CC BY-NC-SA 4.0) and MASC through IEEE DataPort. So that the notebook can run without an account, it uses a small **community-redistributed subset of SADA2022 on the Hugging Face Hub** (`SarahUssama/sada-arabic-test-dataset-sample`, 496 segments with dialect, gender, environment and transcript fields). Verify the licence on the dataset card before use, and prefer the official release for any published result. The notebook also accepts your own audio plus reference transcripts.

## Requirements

Runs on the free CPU tier with `openai/whisper-small` (about 10 minutes of audio takes roughly 15 to 40 minutes on CPU). A T4 GPU is recommended for `openai/whisper-large-v3`.

## Testing status

Executed end to end on CPU with `openai/whisper-small` and the Hub subset. The Label Studio import itself (a desktop or server application) was not run here; the exported files follow the Label Studio task and config formats. The manual tagging step is, by definition, left to the reader.


<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/solutions/Chapter_06_Exercise_05.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [1]:
!pip install -q "transformers>=4.40" torch jiwer soundfile pandas pyarrow huggingface_hub camel-tools
!camel_data -i morphology-db-msa-r13

No new packages will be installed.


## 1. Build a five-minute sample per dialect

We take roughly five minutes of **Saudi** segments and five minutes of **Egyptian** segments from the subset (segments are drawn in the order they appear so that whole programme extracts stay together).

In [2]:
import io, os, re, unicodedata, numpy as np, pandas as pd, soundfile as sf, pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

MINUTES_PER_DIALECT = float(os.environ.get("MINUTES", "5"))
USE_OWN_DATA = False   # set True and provide sample.csv (file, dialect, reference) + WAVs

if USE_OWN_DATA:
    meta = pd.read_csv("sample.csv")
    def load_audio(row):
        y, sr = sf.read(row["file"]); return y, sr
else:
    path = hf_hub_download("SarahUssama/sada-arabic-test-dataset-sample", "default/train/0000.parquet",
                           repo_type="dataset", revision="refs/convert/parquet")
    pf = pq.ParquetFile(path)
    meta = pf.read(columns=[c for c in pf.schema_arrow.names if c != "audio"]).to_pandas()
    meta["row"] = np.arange(len(meta))
    meta = meta.rename(columns={"SpeakerDialect": "dialect", "GroundTruthText": "reference"})
    # prefer segments labelled Clean (the subset also has Noisy and Music segments; keep those for a robustness run),
    # then keep whole segments in corpus order until about MINUTES_PER_DIALECT minutes per dialect
    PREFER_CLEAN = True
    if PREFER_CLEAN:
        meta = meta[meta["Environment"].astype(str).str.startswith("Clean")]
    keep = []
    for dial, g in meta.groupby("dialect", sort=False):
        cum = g["SegmentLength"].cumsum()
        keep.append(g[cum <= MINUTES_PER_DIALECT*60])
    meta = pd.concat(keep).reset_index(drop=True)
    audio_col = pf.read(columns=["audio"]).to_pandas()["audio"]
    def load_audio(row):
        a = audio_col.iloc[int(row["row"])]
        y, sr = sf.read(io.BytesIO(a["bytes"])); return y, sr

print(meta.groupby("dialect")["SegmentLength"].agg(["count", "sum"]).rename(columns={"sum": "seconds"}))

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


          count     seconds
dialect                    
Egyptian     62  188.701151
Saudi        91  298.964349


## 2. Baseline Whisper transcription

`openai/whisper-small` by default (CPU-friendly). On a T4 GPU switch to `openai/whisper-large-v3` for the baseline that most papers use; the audit workflow is identical.

In [3]:
import torch, warnings, transformers, librosa
warnings.filterwarnings("ignore"); transformers.logging.set_verbosity_error()
from transformers import pipeline
MODEL = os.environ.get("ASR_MODEL", "openai/whisper-small")
asr = pipeline("automatic-speech-recognition", model=MODEL, device=0 if torch.cuda.is_available() else -1)
hyps = []
for _, row in meta.iterrows():
    y, sr = load_audio(row)
    if y.ndim > 1: y = y.mean(axis=1)
    if sr != 16000: y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=16000)
    # cap the output length to the audio length (about 6 tokens per second is generous for Arabic) so that a
    # hallucination loop cannot run to Whisper's 448-token limit; repetition is also penalized
    max_tok = int(min(440, 6*len(y)/16000 + 10))
    out = asr({"raw": y.astype(np.float32), "sampling_rate": 16000},
              generate_kwargs={"language": "arabic", "task": "transcribe", "max_new_tokens": max_tok,
                               "no_repeat_ngram_size": 4})
    hyps.append(out["text"].strip())
meta["hypothesis"] = hyps
meta[["dialect", "reference", "hypothesis"]].head(6)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:  29%|██▉       | 139/479 [00:00<00:00, 1357.31it/s]

Loading weights:  57%|█████▋    | 275/479 [00:00<00:00, 1340.65it/s]

Loading weights:  86%|████████▌ | 410/479 [00:00<00:00, 1181.03it/s]

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1217.11it/s]

,dialect,reference,hypothesis
0,Egyptian,وأنا بشجع كل ست مستقلة وقوية ومسيطرة.,وانا بشجع كل ست مستقلة قوية ومسيطر
1,Egyptian,أنا آسفة جدا، أنا مش عارفة أرد على كل الكومنت ...,أنا أسفة جداً أنا مش عارفة رودة على كل الكومنت...
2,Egyptian,إن شاء الله قريب قوي حتلاقيني لايف تاني معاكو....,إن شاء الله ورايب قوي حتلاقوني ليفتاني معك باي
3,Egyptian,ما قلتليش رايحين فين أنا عندي اجتماع في الشركة,والتي ليش مهم فيه أنا أنا لديك دمار الشركة
4,Egyptian,هقول لك وإحنا في الطريق,هقول لك وحن في الطريق
5,Egyptian,ليه هو سر؟,دي هو سر


## 3. Per-dialect WER (the single number)

Normalization: NFC, diacritics and tatweel removed, punctuation removed, alif forms unified, ة/ه and ى/ي unified, Arabic-Indic digits to Western. Diacritics are not scored. The same function is applied to both sides.

In [4]:
import jiwer
DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
PUNCT = re.compile(r"[\u060C\u061B\u061F\u066A-\u066D\u06D4!-/:-@\[-`{-~\u201C\u201D\u2018\u2019\u00AB\u00BB]")
def normalize(t):
    t = unicodedata.normalize("NFC", str(t)).replace("\u0640", "")
    t = DIAC.sub("", t); t = PUNCT.sub(" ", t)
    t = re.sub("[أإآٱ]", "ا", t).replace("ة", "ه").replace("ى", "ي")
    t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    return re.sub(r"\s+", " ", t).strip()

meta["ref_n"] = meta["reference"].map(normalize); meta["hyp_n"] = meta["hypothesis"].map(normalize)
meta = meta[meta["ref_n"].str.len() > 0]
summary = []
for dial, g in meta.groupby("dialect"):
    o = jiwer.process_words(g["ref_n"].tolist(), g["hyp_n"].tolist())
    N = sum(len(r.split()) for r in g["ref_n"])
    summary.append({"dialect": dial, "segments": len(g), "seconds": round(g["SegmentLength"].sum() if "SegmentLength" in g else 0),
                    "N words": N, "S": o.substitutions, "D": o.deletions, "I": o.insertions, "WER %": round(100*o.wer, 1)})
o_all = jiwer.process_words(meta["ref_n"].tolist(), meta["hyp_n"].tolist())
summary.append({"dialect": "POOLED", "segments": len(meta), "seconds": round(meta["SegmentLength"].sum()) if "SegmentLength" in meta else 0,
                "N words": sum(len(r.split()) for r in meta["ref_n"]), "S": o_all.substitutions, "D": o_all.deletions,
                "I": o_all.insertions, "WER %": round(100*o_all.wer, 1)})
print("model:", MODEL); pd.DataFrame(summary)

model: openai/whisper-small


,dialect,segments,seconds,N words,S,D,I,WER %
0,Egyptian,62,189,490,246,73,14,68.0
1,Saudi,91,299,687,306,106,30,64.3
2,POOLED,153,488,1177,552,179,44,65.8


## 4. Word-level alignment and a first-pass automatic pre-tag

For the manual audit we need every error as a row: reference word, hypothesis word, operation. To speed the human pass we add an **automatic pre-tag** that is only a hint: if the hypothesis word is a valid MSA word according to the CAMeL Tools analyzer and the reference word is *not* in the MSA lexicon, the substitution is flagged as a *candidate* MSA shift. The annotator confirms or overrides every tag; the heuristic is not the audit.

In [5]:
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
analyzer = Analyzer(MorphologyDB.builtin_db("calima-msa-r13"))
def in_msa_lexicon(w):
    return len(analyzer.analyze(w)) > 0

rows = []
for i, r in meta.iterrows():
    out = jiwer.process_words(r["ref_n"], r["hyp_n"])
    ref_w, hyp_w = r["ref_n"].split(), r["hyp_n"].split()
    for ch in out.alignments[0]:
        if ch.type == "equal": continue
        for k in range(max(ch.ref_end_idx - ch.ref_start_idx, ch.hyp_end_idx - ch.hyp_start_idx)):
            rw = ref_w[ch.ref_start_idx + k] if ch.ref_start_idx + k < ch.ref_end_idx else ""
            hw = hyp_w[ch.hyp_start_idx + k] if ch.hyp_start_idx + k < ch.hyp_end_idx else ""
            pretag = ""
            if len(hyp_w) > 2*len(ref_w) + 3 and ch.type == "insert":
                pretag = "candidate hallucination (insertion run)"
            if ch.type == "substitute" and rw and hw:
                if in_msa_lexicon(hw) and not in_msa_lexicon(rw): pretag = "candidate MSA shift"
                elif len(set(rw) ^ set(hw)) <= 2: pretag = "candidate spelling variant"
            rows.append({"segment": i, "dialect": r["dialect"], "op": ch.type, "ref_word": rw, "hyp_word": hw, "pretag": pretag,
                         "reference": r["reference"], "hypothesis": r["hypothesis"]})
errors = pd.DataFrame(rows)
print(len(errors), "error rows")
errors.groupby(["dialect", "op"]).size().unstack(fill_value=0)

775 error rows


op,delete,insert,substitute
dialect,,,
Egyptian,73,14,246
Saudi,106,30,306


In [6]:
errors[errors["pretag"] == "candidate MSA shift"][["dialect", "ref_word", "hyp_word", "reference"]].head(15)

,dialect,ref_word,hyp_word,reference
37,Egyptian,بيتقال,بتال,نفس الكلام إلي بيتقال كل مرة
42,Egyptian,متجوزين,جوزين,إحنا متجوزين بقالنا ثمان سنين
59,Egyptian,بالظبط,بزبط,نفس الكلام بالظبط يا خسارت المتين ريال إللي ...
61,Egyptian,خسارت,المتر,نفس الكلام بالظبط يا خسارت المتين ريال إللي ...
91,Egyptian,مالجيزه,شكرا,لا أنا مالجيزة شكرا متشكرين أوي يلا حبيبة
92,Egyptian,متشكرين,اولا,لا أنا مالجيزة شكرا متشكرين أوي يلا حبيبة
94,Egyptian,يلا,الله,لا أنا مالجيزة شكرا متشكرين أوي يلا حبيبة
111,Egyptian,ارحمي,رحمي,يا حبيبية إرحمي نفسك بقى ما تخافيش مش هتجوز ...
114,Egyptian,عليكي,عليك,يا حبيبية إرحمي نفسك بقى ما تخافيش مش هتجوز ...
135,Egyptian,هوصلك,وصلك,خلاص هوصلك البيت يلا حبيبة


## 5. Export to Label Studio for the manual audit

The cell writes `label_studio_tasks.json` (one task per segment: audio path, reference, hypothesis, and the error rows as pre-annotations) and `label_studio_config.xml` (the labelling interface). In Label Studio: create a project, paste the XML into *Labeling Interface > Code*, then *Import* the JSON. Each error row gets exactly one of the tags below; the annotator listens to the audio before tagging.

| Tag | Meaning |
|---|---|
| `acoustic` | genuine misrecognition: the output does not sound like what was said |
| `msa_shift` | the dialect word was rendered as its formal (MSA) equivalent: same meaning, different lexeme |
| `spelling_variant` | same word, acceptable alternative dialect spelling (a normalization or CODA issue, not a recognition error) |
| `deletion` / `insertion` | dropped or added material |
| `hallucination` | a run of inserted words with no counterpart in the audio (Whisper's repetition or invention failure, Section 6.9) |
| `reference_error` | the reference transcript is wrong |

In [7]:
import json
os.makedirs("ls_audio", exist_ok=True)
tasks = []
for i, r in meta.iterrows():
    y, sr = load_audio(r); wav = f"ls_audio/seg_{i:04d}.wav"; sf.write(wav, y, sr)
    seg_err = errors[errors["segment"] == i]
    tasks.append({"data": {"audio": wav, "dialect": r["dialect"], "reference": r["reference"], "hypothesis": r["hypothesis"],
                           "errors": "\n".join(f"{e.op}: {e.ref_word} -> {e.hyp_word} [{e.pretag}]" for e in seg_err.itertuples())}})
json.dump(tasks, open("label_studio_tasks.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

config = '''<View>
  <Header value="Dialect: $dialect"/>
  <Audio name="audio" value="$audio"/>
  <Header value="Reference"/><Text name="ref" value="$reference"/>
  <Header value="Whisper hypothesis"/><Text name="hyp" value="$hypothesis"/>
  <Header value="Error rows (op: reference -> hypothesis [pretag])"/><Text name="err" value="$errors"/>
  <Header value="Tag EACH error row, in order"/>
  <Choices name="tags" toName="err" choice="multiple" showInline="true">
    <Choice value="acoustic"/><Choice value="msa_shift"/><Choice value="spelling_variant"/>
    <Choice value="deletion"/><Choice value="insertion"/><Choice value="hallucination"/><Choice value="reference_error"/>
  </Choices>
  <TextArea name="notes" toName="err" placeholder="one tag per error row, e.g. 1 msa_shift; 2 acoustic" rows="3"/>
</View>'''
open("label_studio_config.xml", "w").write(config)
print(len(tasks), "tasks written to label_studio_tasks.json; interface in label_studio_config.xml")
try:
    from google.colab import files
    files.download("label_studio_tasks.json"); files.download("label_studio_config.xml")
except ImportError:
    pass

153 tasks written to label_studio_tasks.json; interface in label_studio_config.xml


## 6. After the manual pass: split the WER by cause

Paste the tag counts from Label Studio into the dictionary below (per dialect), or fill it from the notebook's `errors` table once you have added a `tag` column. The cell recomputes what the WER would be if MSA shifts and spelling variants were *not* counted, which is the number that reflects acoustic recognition alone.

In [8]:
# Example structure (replace with your annotated counts). Values here are the automatic pre-tags, NOT a manual audit.
per_dialect = {}
for dial, g in meta.groupby("dialect"):
    N = sum(len(r.split()) for r in g["ref_n"])
    e = errors[errors["dialect"] == dial]
    n_shift = int((e["pretag"] == "candidate MSA shift").sum())
    n_spell = int((e["pretag"] == "candidate spelling variant").sum())
    n_hall = int((e["pretag"] == "candidate hallucination (insertion run)").sum())
    n_all = len(e)
    per_dialect[dial] = {"N": N, "errors": n_all, "pretag MSA shift": n_shift, "pretag spelling": n_spell,
                         "pretag hallucination": n_hall, "WER %": round(100*n_all/N, 1),
                         "WER excluding shifts+spelling+hallucination %": round(100*(n_all-n_shift-n_spell-n_hall)/N, 1)}
pd.DataFrame(per_dialect).T

,N,errors,pretag MSA shift,pretag spelling,pretag hallucination,WER %,WER excluding shifts+spelling+hallucination %
Egyptian,490.0,333.0,24.0,102.0,10.0,68.0,40.2
Saudi,687.0,442.0,27.0,123.0,0.0,64.3,42.5


## 7. Discussion template

> Per-dialect WER (model _, normalization as above): Saudi _ %, Egyptian _ %, pooled _ %. Of the Saudi errors, _ % were tagged `msa_shift` and _ % `spelling_variant` after manual audit; excluding those, the acoustic WER is _ %. The single WER therefore overstates how often the model *misheard* the speaker and hides that a large share of its "errors" are systematic translations of dialect words into MSA equivalents (for example وين rendered as أين, أبغى as أريد), which a WER counts identically to a genuine acoustic confusion. Conversely, the pooled number hides the gap between the two dialects because the larger sample dominates it.